# Painel de Repasse de Preço via Inflação
## IPCA: Alimentação fora do domicílio (2012–2026)

**Objetivo:** Simular o impacto da variação mensal do IPCA (a nálise parte somente do subgrupo Alimentação fora do domicílio)
no repasse de preço e margem de um negócio hipotético de food service.

**Fonte:** API IBGE/SIDRA — Tabelas 1419 (2012–2019) e 7060 (2020–2026)  
**Destino:** BigQuery - dataset `inflacao_br`  
**Autor:** João Madeira  

## 00. Imports e Configuração

In [15]:
import requests
import pandas as pd
from google.colab import auth
from google.cloud import bigquery

PROJECT_ID = "port-joaomadeira"
DATASET = "projeto_inflacao"
TABLE = "ipca_alimentacao_fora_domicilio"

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

print("✅ Autenticado e pronto.")

✅ Autenticado e pronto.


---
## 01. Ingestão -> API IBGE/SIDRA

O IBGE divide a série histórica do IPCA por subgrupo em tabelas separadas por período:
- **Tabela 1419** → 2012 a 2019
- **Tabela 7060** → 2020 a 2026

Criei uma função única que recebe o número da tabela e retorna um DataFrame padronizado.
Isso evita repetir o mesmo bloco de código pra cada tabela.

**Parâmetros fixos:**
- Variável `63` = IPCA variação mensal
- Classificação `315`, categoria `7432` = Alimentação fora do domicílio
- Localidade `N1` = Brasil (nível nacional)

In [16]:
def busca_ipca_subgrupo(tabela, variavel=63, categoria=7432):
    url = (
        f"https://servicodados.ibge.gov.br/api/v3/agregados/{tabela}"
        f"/periodos/all"
        f"/variaveis/{variavel}"
        f"?localidades=N1[all]"
        f"&classificacao=315[{categoria}]"
    )
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception(f"Erro na API — tabela {tabela}: status {response.status_code}")

    serie = response.json()[0]['resultados'][0]['series'][0]['serie']

    df = pd.DataFrame([
        {'periodo': k, 'variacao_mensal': float(v)}
        for k, v in serie.items()
    ])
    df['periodo'] = pd.to_datetime(df['periodo'], format='%Y%m')
    return df


df_2012 = busca_ipca_subgrupo(1419)  # 2012–2019
df_2020 = busca_ipca_subgrupo(7060)  # 2020–2026

print(f"Tabela 1419: {len(df_2012)} linhas ({df_2012['periodo'].min().date()} → {df_2012['periodo'].max().date()})")
print(f"Tabela 7060: {len(df_2020)} linhas ({df_2020['periodo'].min().date()} → {df_2020['periodo'].max().date()})")

Tabela 1419: 96 linhas (2012-01-01 → 2019-12-01)
Tabela 7060: 79 linhas (2020-01-01 → 2026-07-01)


---
## 02. Tratamento — Consolidação e Colunas Derivadas

Aqui fiz três coisas:
1. **Concat + dedup:** une as duas tabelas e remove eventuais períodos duplicados
2. **Colunas de calendário:** ano e mês separados — úteis pra filtros no Power BI
3. **Colunas analíticas:**
   - `variacao_acum_12m` → inflação acumulada nos últimos 12 meses (rolling sum)
   - `repasse_lag1` → variação do mês anterior (simula repasse com 1 mês de atraso)
   - `repasse_lag2` → variação de 2 meses atrás (cenário alternativo)
   - `margem_absorvida` → diferença entre custo (variação mensal) e repasse (lag1) — quanto o negócio não repassou

**Nota sobre os NaN:** os primeiros 11 meses de `variacao_acum_12m` e os primeiros 1–2 meses
de repasse serão NaN por definição — não há 12 meses anteriores pra calcular. Isso é esperado.

In [17]:
df = (
    pd.concat([df_2012, df_2020], ignore_index=True)
    .drop_duplicates(subset='periodo')
    .sort_values('periodo')
    .reset_index(drop=True)
)

# Colunas de calendário
df['ano'] = df['periodo'].dt.year
df['mes'] = df['periodo'].dt.month

# Acumulado 12 meses (soma rolling)
df['variacao_acum_12m'] = df['variacao_mensal'].rolling(12).sum().round(2)

# Repasse simulado com lag (o negócio ajusta o preço com atraso)
df['repasse_lag1'] = df['variacao_mensal'].shift(1)  # cenário base
df['repasse_lag2'] = df['variacao_mensal'].shift(2)  # cenário alternativo

# Margem absorvida = custo do mês - o que foi repassado (lag1)
# Positivo = o negócio absorveu custo (margem comprimida)
# Negativo = o negócio repassou mais do que o custo subiu
df['margem_absorvida_lag1'] = (df['variacao_mensal'] - df['repasse_lag1']).round(2)
df['margem_absorvida_lag2'] = (df['variacao_mensal'] - df['repasse_lag2']).round(2)

# Metadados
df['subgrupo'] = 'Alimentação fora do domicílio'
df['indice']   = 'IPCA'

print(df.shape)
print(df.dtypes)
df.tail(5)

(175, 11)
periodo                  datetime64[ns]
variacao_mensal                 float64
ano                               int32
mes                               int32
variacao_acum_12m               float64
repasse_lag1                    float64
repasse_lag2                    float64
margem_absorvida_lag1           float64
margem_absorvida_lag2           float64
subgrupo                         object
indice                           object
dtype: object


,periodo,variacao_mensal,ano,mes,variacao_acum_12m,repasse_lag1,repasse_lag2,margem_absorvida_lag1,margem_absorvida_lag2,subgrupo,indice
170,2026-03-01,0.61,2026,3,6.34,0.34,0.55,0.27,0.06,Alimentação fora do domicílio,IPCA
171,2026-04-01,0.59,2026,4,6.13,0.61,0.34,-0.02,0.25,Alimentação fora do domicílio,IPCA
172,2026-05-01,0.49,2026,5,6.04,0.59,0.61,-0.10,-0.12,Alimentação fora do domicílio,IPCA
173,2026-06-01,0.15,2026,6,5.73,0.49,0.59,-0.34,-0.44,Alimentação fora do domicílio,IPCA
174,2026-07-01,0.55,2026,7,5.41,0.15,0.49,0.40,0.06,Alimentação fora do domicílio,IPCA


---
## 03. Validação

Antes de carregar no BigQuery, checo:
- Shape esperado
- Período correto (sem gaps)
- Nulos aceitáveis (só nas colunas de rolling/lag, nos primeiros meses)

In [18]:
print("=== Shape ===")
print(df.shape)

print("\n=== Período ===")
print(f"Início: {df['periodo'].min().date()}")
print(f"Fim:    {df['periodo'].max().date()}")
print(f"Meses únicos: {df['periodo'].nunique()}")

print("\n=== Nulos ===")
print(df.isnull().sum())

print("\n=== Estatísticas básicas ===")
print(df['variacao_mensal'].describe().round(2))

=== Shape ===
(175, 11)

=== Período ===
Início: 2012-01-01
Fim:    2026-07-01
Meses únicos: 175

=== Nulos ===
periodo                   0
variacao_mensal           0
ano                       0
mes                       0
variacao_acum_12m        11
repasse_lag1              1
repasse_lag2              2
margem_absorvida_lag1     1
margem_absorvida_lag2     2
subgrupo                  0
indice                    0
dtype: int64

=== Estatísticas básicas ===
count    175.00
mean       0.55
std        0.32
min       -0.29
25%        0.32
50%        0.53
75%        0.77
max        1.31
Name: variacao_mensal, dtype: float64


---
## 04. Carga — BigQuery

Carrego o DF tratado na tabela
`ipca_alimentacao_fora_domicilio`
do dataset
`inflacao_br`.

`WRITE_TRUNCATE` sobrescreve a tabela a cada execução, garante estabilização: se rodar duas vezes, não duplica dado.

In [19]:
df_load = df.copy()
df_load['periodo'] = df_load['periodo'].dt.date
schema = [
    bigquery.SchemaField("periodo",               "DATE"),
    bigquery.SchemaField("variacao_mensal",        "FLOAT64"),
    bigquery.SchemaField("ano",                   "INT64"),
    bigquery.SchemaField("mes",                   "INT64"),
    bigquery.SchemaField("variacao_acum_12m",      "FLOAT64"),
    bigquery.SchemaField("repasse_lag1",           "FLOAT64"),
    bigquery.SchemaField("repasse_lag2",           "FLOAT64"),
    bigquery.SchemaField("margem_absorvida_lag1",  "FLOAT64"),
    bigquery.SchemaField("margem_absorvida_lag2",  "FLOAT64"),
    bigquery.SchemaField("subgrupo",              "STRING"),
    bigquery.SchemaField("indice",                "STRING"),
]

table_ref  = f"{PROJECT_ID}.{DATASET}.{TABLE}"
job_config = bigquery.LoadJobConfig(
    schema=schema,
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(df_load, table_ref, job_config=job_config)
job.result()

n = client.get_table(table_ref).num_rows
print(f"✅ {n} linhas carregadas em {table_ref}")

✅ 175 linhas carregadas em port-joaomadeira.projeto_inflacao.ipca_alimentacao_fora_domicilio
